# KumoTabular (Nvidia) — Serving endpoint with context in Lakebase

Serves **KumoTabular** from one GPU serving endpoint for many prediction tasks. The context rows of every task
live in **Lakebase**. The endpoint fetches them when a task is first used and keeps the fitted context on the GPU.

> **License note:** Kumo is released under a [commercially permissive license](https://openmdw.ai/license/1-1/)

An in-context model needs labelled rows, the context, next to the rows it should predict. There are three places
to keep that context when you serve the model.

| Where the context lives | One endpoint serves | To change the context | Limit |
|---|---|---|---|
| In every request | any task | send other rows | The request must stay under the [payload limit](https://docs.databricks.com/aws/en/machine-learning/model-serving/model-serving-limits) of 16 MB, and the model fits the context on every call |
| Inside the model artifact | one task | log a new model version and update the endpoint | One endpoint per task |
| **In Lakebase (this notebook)** | any task that has a context table | write a new version to a Delta table and sync it | The first request for a task version pays for the fit |

The [TabICL serving notebook](../../tabicl/notebooks/03_inference_on_serving_endpoint.ipynb) shows the first two.
This notebook shows the third with the five KumoTabular tasks of this repo, three classification tasks and two
regression tasks, behind a single endpoint.

**How it works**

1. The context rows of each task are written to a Delta table, with a version number. A second small table,
   the registry, holds the active version of each task.
2. Both tables are synced to Lakebase.
3. A request names its task. The endpoint reads the active version from the registry. If that version is not
   fitted yet, it fetches the rows, fits them once, and keeps the result on the GPU.
4. To refresh a context, write the rows of a new version, sync, and then update the registry. No model is
   logged and no endpoint is updated.

**Compute:** Serverless CPU environment v5. The serving endpoint GPU is used for inference.

**Prerequisites**

- Run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.
- A Lakebase database instance, and the `databricks_superuser` role on it for the user who runs this notebook.
  See the Lakebase note in the Configuration section.

In [0]:
%pip install "mlflow>=3.3.2" "databricks-sdk>=0.81.0" "psycopg[binary]>=3.2" "huggingface_hub>=0.23" --quiet

In [0]:
dbutils.library.restartPython()

## Configuration

Catalog/schema must match the shared data-prep notebook. `common/` is added to the
path so we can import the shared evaluation helpers.

**Lakebase.** Set the `lakebase_instance` widget to the name of your Lakebase database instance. The endpoint
signs in to Lakebase through the MLflow resource `DatabricksLakebase`, so no secret is created or stored. That
resource takes the name of a database instance. Use an instance that `databricks database get-database-instance
<name>` can find. A Lakebase project that was created only through the Postgres API is not covered by this
resource.

In [0]:
import os, sys

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation, imported from common/config.py)
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Lakebase database instance that holds the contexts, and the Postgres database inside it
dbutils.widgets.text("lakebase_instance", "tabular-fm", "Lakebase database instance")
LAKEBASE_INSTANCE = dbutils.widgets.get("lakebase_instance")
LAKEBASE_DATABASE = "databricks_postgres"

# Delta tables with the contexts, and their synced copies in Lakebase
CONTEXT_ROWS_TABLE = "kumo_context_rows"
CONTEXT_REGISTRY_TABLE = "kumo_context_registry"
ROWS_SYNCED_TABLE = "kumo_context_rows_synced"
REGISTRY_SYNCED_TABLE = "kumo_context_registry_synced"

# Model and endpoint
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.kumo_tabular_lakebase_context"
SERVING_ENDPOINT = "kumo_tabular_lakebase_context"
MODEL_SIZE = "large"        # "small", "medium" or "large", as in the other KumoTabular notebooks
NUM_ESTIMATORS = 8

# MLflow experiments (shared naming convention across vendors)
current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME_SERVING_ENDPOINTS = f"/Users/{current_user}/tabular-fm-databricks-serving-endpoints"
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"common/ path:   {COMMON_PATH}  (exists={os.path.isdir(COMMON_PATH)})")
print(f"Lakebase:       instance {LAKEBASE_INSTANCE}, database {LAKEBASE_DATABASE}")

## Import libraries

In [0]:
import json
import time
import uuid

import numpy as np
import pandas as pd
import mlflow
import psycopg
from huggingface_hub import hf_hub_download
from mlflow.models import ModelSignature
from mlflow.models.resources import DatabricksLakebase
from mlflow.types import Schema, ColSpec, DataType, ParamSchema, ParamSpec

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.database import (
    NewPipelineSpec,
    SyncedDatabaseTable,
    SyncedTableSchedulingPolicy,
    SyncedTableSpec,
)
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    EndpointStateConfigUpdate,
    EndpointStateReady,
    ServedEntityInput,
    ServingModelWorkloadType,
)

from evaluation import (
    split_xy, classification_metrics, regression_metrics,
    log_result, RESULTS_TABLE,
)

mlflow.set_registry_uri("databricks-uc")
w = WorkspaceClient()

## Publish the contexts as Delta tables

The five tasks are the ones of the KumoTabular classification and regression notebooks, with the same
train/test split. The train part of each task becomes its context. The test part is used later to query the
endpoint.

The tasks have different columns, so one row of context is stored as a JSON document.

| Table | Columns | Purpose |
|---|---|---|
| `kumo_context_rows` | `task`, `version`, `row_id`, `features` (JSON), `target` | The context rows of every task and version |
| `kumo_context_registry` | `task`, `version`, `problem_type`, `n_rows` | The active version of every task |

In [0]:
TASKS = {
    "supplier_delay_risk": dict(table="supplier_delay_risk_train", target="is_delayed",
                                problem_type="binary_classification", test_size=0.2, stratify=True),
    "material_shortage": dict(table="material_shortage_train", target="shortage_risk",
                              problem_type="multiclass_classification", test_size=0.3, stratify=True),
    "otif_risk": dict(table="otif_risk_train", target="otif_risk",
                      problem_type="multiclass_classification", test_size=0.3, stratify=True),
    "price_elasticity": dict(table="price_elasticity_train", target="price_elasticity",
                             problem_type="regression", test_size=0.2, stratify=False),
    "supplier_lead_time": dict(table="supplier_lead_time_train", target="actual_lead_time_days",
                               problem_type="regression", test_size=0.2, stratify=False),
}


def to_records(X):
    """Rows as JSON-safe dicts, with floats kept exactly."""
    return json.loads(X.to_json(orient="records", double_precision=15))


def context_frames(task, version, X, y, problem_type):
    """The rows of one context version, and its registry row."""
    kind = "regression" if problem_type == "regression" else "classification"
    rows = pd.DataFrame({
        "task": task,
        "version": version,
        "row_id": np.arange(len(X)),
        "features": [json.dumps(r) for r in to_records(X)],
        "target": [str(v) for v in y],
    })
    registry = pd.DataFrame([{"task": task, "version": version, "problem_type": kind, "n_rows": len(X)}])
    return rows, registry

In [0]:
splits, rows_parts, registry_parts = {}, [], []
for task, t in TASKS.items():
    df = spark.table(t["table"]).toPandas()
    X_train, X_test, y_train, y_test = split_xy(
        df, target=t["target"], test_size=t["test_size"], stratify=t["stratify"]
    )
    splits[task] = (X_train, X_test, y_train, y_test)
    rows, registry = context_frames(task, 1, X_train, y_train, t["problem_type"])
    rows_parts.append(rows)
    registry_parts.append(registry)

for table, parts in ((CONTEXT_ROWS_TABLE, rows_parts), (CONTEXT_REGISTRY_TABLE, registry_parts)):
    (spark.createDataFrame(pd.concat(parts, ignore_index=True))
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table))
    # A triggered sync reads the change data feed of its source table
    spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

display(spark.table(CONTEXT_REGISTRY_TABLE).orderBy("task"))

## Sync the context tables to Lakebase

A synced table copies a Delta table into Postgres and keeps it up to date. The sync runs when it is triggered,
so a new context version reaches the endpoint only when you ask for it.

In [0]:
ONLINE_STATES = {"SYNCED_TABLE_ONLINE", "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE", "SYNCED_TABLE_ONLINE_TRIGGERED_UPDATE"}
FAILED_STATES = {"SYNCED_TABLE_OFFLINE_FAILED", "SYNCED_TABLE_ONLINE_PIPELINE_FAILED"}

LAKEBASE_HOST = w.database.get_database_instance(LAKEBASE_INSTANCE).read_write_dns


def sync_status(name):
    status = w.database.get_synced_database_table(name).data_synchronization_status
    state = getattr(status, "detailed_state", None)
    return status, getattr(state, "value", str(state))


def wait_online(name, timeout_s=3600):
    start = time.time()
    while True:
        status, state = sync_status(name)
        if state in ONLINE_STATES:
            return status
        if state in FAILED_STATES or time.time() - start > timeout_s:
            raise RuntimeError(f"Synced table {name}: {state}")
        time.sleep(20)


def sync_now(name, timeout_s=3600):
    """Trigger one sync of a synced table and wait until it has finished."""
    pipeline_id = wait_online(name).pipeline_id
    update_id = w.pipelines.start_update(pipeline_id=pipeline_id).update_id
    start = time.time()
    while True:
        state = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id).update.state.value
        if state == "COMPLETED":
            return
        if state in ("FAILED", "CANCELED") or time.time() - start > timeout_s:
            raise RuntimeError(f"Sync of {name}: {state}")
        time.sleep(10)


def create_synced_table(source, target, primary_key):
    """Create the synced table, or sync it again if it already exists."""
    name = f"{CATALOG}.{SCHEMA}.{target}"
    try:
        w.database.get_synced_database_table(name)
        sync_now(name)
        print(f"{name}: synced again")
    except NotFound:
        w.database.create_synced_database_table(SyncedDatabaseTable(
            name=name,
            database_instance_name=LAKEBASE_INSTANCE,
            logical_database_name=LAKEBASE_DATABASE,
            spec=SyncedTableSpec(
                source_table_full_name=f"{CATALOG}.{SCHEMA}.{source}",
                primary_key_columns=primary_key,
                scheduling_policy=SyncedTableSchedulingPolicy.TRIGGERED,
                create_database_objects_if_missing=True,
                new_pipeline_spec=NewPipelineSpec(storage_catalog=CATALOG, storage_schema=SCHEMA),
            ),
        ))
        wait_online(name)
        print(f"{name}: created")
    return name


# Rows first, registry second: the endpoint must never see a version whose rows are not there yet
ROWS_SYNCED = create_synced_table(CONTEXT_ROWS_TABLE, ROWS_SYNCED_TABLE, ["task", "version", "row_id"])
REGISTRY_SYNCED = create_synced_table(CONTEXT_REGISTRY_TABLE, REGISTRY_SYNCED_TABLE, ["task"])

Read the registry back from Postgres, the way the endpoint will.

In [0]:
def lakebase_sql(query, args=None, fetch=True):
    """Run one statement in Lakebase as the current user, with a short-lived database credential."""
    token = w.database.generate_database_credential(
        instance_names=[LAKEBASE_INSTANCE], request_id=str(uuid.uuid4())
    ).token
    with psycopg.connect(host=LAKEBASE_HOST, dbname=LAKEBASE_DATABASE, user=current_user, password=token,
                         sslmode="require", autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(query, args)
            return cur.fetchall() if fetch else None


def wait_for_rows(timeout_s=600):
    """A sync that reports done can reach Postgres a moment later, so check the row counts there."""
    start = time.time()
    while True:
        registry = lakebase_sql(
            f'SELECT task, version, problem_type, n_rows FROM "{SCHEMA}"."{REGISTRY_SYNCED_TABLE}" ORDER BY task'
        )
        counts = dict(((t, v), n) for t, v, n in lakebase_sql(
            f'SELECT task, version, count(*) FROM "{SCHEMA}"."{ROWS_SYNCED_TABLE}" GROUP BY task, version'
        ))
        if registry and all(counts.get((task, version)) == n_rows for task, version, _, n_rows in registry):
            return pd.DataFrame(registry, columns=["task", "version", "problem_type", "n_rows"])
        if time.time() - start > timeout_s:
            raise RuntimeError(f"Lakebase does not hold every context row yet: {registry} {counts}")
        time.sleep(10)


display(wait_for_rows())

## Create the serving endpoint

### The served model

The model is written to a file and logged as [models from code](https://mlflow.org/docs/latest/ml/model/models-from-code/).
It holds no context. It loads the KumoTabular weights for classification and for regression once. For each
task it keeps one fitted context, and all tasks of one kind share the same weights.

A request names its task in a `__task__` column, or in `params={"task": ...}`. The answer has the same columns
for every task:

- `prediction`: the predicted class, or the predicted value (the median of the predicted distribution)
- `details`: a JSON document. Class probabilities for classification, the 5%, 50% and 95% quantiles for regression
- `task`, `context_version`, `context_cached`, `server_ms`: which context answered, whether it was already
  fitted, and the time spent inside the model

In [0]:
%%writefile /tmp/kumo_lakebase_context_model.py
"""KumoTabular behind one serving endpoint, with the context of every task in Lakebase.

A request names a task. The model reads the task's active context version from a small registry table in
Lakebase. If that version is not fitted yet, it fetches the context rows, fits them once, and keeps the fitted
context on the GPU. Later requests for the task only pay for the prediction.
"""

import collections
import copy
import importlib.util
import json
import threading
import time

import mlflow
import pandas as pd


class KumoTabularLakebaseContext(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with open(context.artifacts["settings"]) as fh:
            self.settings = json.load(fh)
        self.artifacts = context.artifacts
        self.ready = False
        # MLflow also calls this while logging. The logging notebook has no model library, so the weights are
        # loaded here only on the endpoint.
        if importlib.util.find_spec("sdm") is not None:
            self._load()

    def _load(self):
        import sdm
        import torch

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.base = {}
        for kind in ("classification", "regression"):
            model = sdm.models.KumoTabular(task=kind, size=self.settings["size"], pretrained=False, device=self.device)
            state = torch.load(self.artifacts[kind], map_location=self.device, weights_only=True)
            model.models[kind].load_state_dict(state, assign=True)
            self.base[kind] = model.eval()
        self.fitted = collections.OrderedDict()  # task -> fitted context, least recently used first
        self.checked = {}  # task -> (time of the last registry read, registry row)
        self.lock = threading.Lock()
        self.connection, self.connected_at, self.user = None, 0.0, None
        self.ready = True

    # ---- Lakebase ----
    def _connect(self):
        import psycopg
        from databricks.sdk import WorkspaceClient

        lb = self.settings["lakebase"]
        w = WorkspaceClient()  # on the endpoint this is the identity created for the declared Lakebase resource
        token = w.database.generate_database_credential(
            instance_names=[lb["instance"]], request_id=f"kumo-{time.time_ns()}"
        ).token
        if self.user is None:
            try:
                self.user = w.current_user.me().user_name
            except Exception:  # the endpoint's token may not reach the user API
                self.user = w.config.client_id
        self.connection = psycopg.connect(host=lb["host"], dbname=lb["database"], user=self.user, password=token,
                                          sslmode="require", autocommit=True, connect_timeout=15)  # fmt: skip
        self.connected_at = time.time()

    def _sql(self, query, args):
        """Run one query. The database credential lasts an hour, so reconnect after 45 minutes or on a failure."""
        for attempt in (0, 1):
            try:
                if self.connection is None or self.connection.closed or time.time() - self.connected_at > 2700:
                    self._connect()
                with self.connection.cursor() as cur:
                    cur.execute(query, args)
                    return cur.fetchall()
            except Exception:
                self.connection = None
                if attempt:
                    raise

    def _registry(self, task):
        seen = self.checked.get(task)
        if seen and time.time() - seen[0] < self.settings["version_check_seconds"]:
            return seen[1]
        lb = self.settings["lakebase"]
        rows = self._sql(
            f'SELECT version, problem_type, n_rows FROM "{lb["schema"]}"."{lb["registry_table"]}" WHERE task = %s',
            (task,),
        )
        if not rows:
            raise ValueError(f"task {task!r} has no context in the registry table")
        reg = {"version": int(rows[0][0]), "kind": rows[0][1], "n_rows": int(rows[0][2])}
        self.checked[task] = (time.time(), reg)
        return reg

    # ---- fit once per task and version ----
    def _fit(self, task, reg):
        import sdm
        import torch

        lb = self.settings["lakebase"]
        rows = self._sql(
            f'SELECT features, target FROM "{lb["schema"]}"."{lb["rows_table"]}" '
            "WHERE task = %s AND version = %s ORDER BY row_id",
            (task, reg["version"]),
        )
        if len(rows) != reg["n_rows"]:
            raise RuntimeError(
                f"{task} version {reg['version']}: {len(rows)} rows in Lakebase, the registry says {reg['n_rows']}. "
                "Sync the rows table before the registry table."
            )
        x = pd.DataFrame([json.loads(features) for features, _ in rows])
        y = pd.DataFrame({"target": [target for _, target in rows]})
        if reg["kind"] == "regression":
            y["target"] = y["target"].astype(float)
            y_stypes = sdm.infer_stypes(y)
        else:
            y_stypes = sdm.infer_stypes(y, overrides={"target": "categorical"})
        stypes = sdm.infer_stypes(x)
        # A shallow copy shares the weights of the base model and holds its own fitted context.
        model = copy.copy(self.base[reg["kind"]])
        model.fit(
            x=sdm.TableTensor.from_pandas(df=x, stypes=stypes, device=self.device),
            y=sdm.TableTensor.from_pandas(df=y, stypes=y_stypes, device=self.device),
            num_estimators=self.settings["num_estimators"],
            estimator_batch_size=None,
            generator=torch.Generator(device=self.device).manual_seed(0),  # the same context gives the same fit
        )
        return {"version": reg["version"], "kind": reg["kind"], "model": model, "stypes": stypes,
                "columns": list(x.columns)}  # fmt: skip

    def predict(self, context, model_input, params=None):
        import sdm

        start = time.time()
        if not self.ready:
            self._load()
        frame = pd.DataFrame(model_input)
        task = (params or {}).get("task") or ""
        if "__task__" in frame.columns:  # a column works from every client, including ai_query
            task = str(frame["__task__"].iloc[0])
            frame = frame.drop(columns="__task__")
        if not task:
            raise ValueError("name the task in a `__task__` column or in params={'task': ...}")
        with self.lock:
            reg = self._registry(task)
            entry = self.fitted.get(task)
            cached = entry is not None and entry["version"] == reg["version"]
            if not cached:
                entry = self.fitted[task] = self._fit(task, reg)
                while len(self.fitted) > self.settings["max_cached_tasks"]:
                    self.fitted.popitem(last=False)  # drop the task that was used longest ago
            self.fitted.move_to_end(task)
            missing = [c for c in entry["columns"] if c not in frame.columns]
            if missing:
                raise ValueError(f"task {task!r} needs the columns {missing}")
            queries = sdm.TableTensor.from_pandas(
                df=frame[entry["columns"]], stypes=entry["stypes"], device=self.device
            )
            out = entry["model"].predict(queries).to_pandas()
        if entry["kind"] == "classification":
            classes = sorted(out.columns, key=str)
            proba = out[classes].to_numpy()
            prediction = [str(classes[i]) for i in proba.argmax(axis=1)]
            details = [json.dumps({str(c): round(float(p), 6) for c, p in zip(classes, row, strict=True)})
                       for row in proba]  # fmt: skip
        else:  # 999 quantiles; the median is the point estimate
            prediction = [str(float(v)) for v in out["q500"]]
            details = [json.dumps({q: round(float(row[q]), 6) for q in ("q050", "q500", "q950")})
                       for _, row in out.iterrows()]  # fmt: skip
        n = len(frame)
        return pd.DataFrame({
            "prediction": prediction,
            "details": details,
            "task": [task] * n,
            "context_version": [entry["version"]] * n,
            "context_cached": [cached] * n,
            "server_ms": [round((time.time() - start) * 1000, 1)] * n,
        })  # fmt: skip


mlflow.models.set_model(KumoTabularLakebaseContext())

### Register model into UC

The weights are downloaded from Hugging Face here and stored with the model, so the endpoint does not need to
reach Hugging Face. The `DatabricksLakebase` resource tells Model Serving that the endpoint may sign in to the
Lakebase instance.

In [0]:
settings = {
    "size": MODEL_SIZE,
    "num_estimators": NUM_ESTIMATORS,
    "max_cached_tasks": 16,          # fitted contexts kept on the GPU; the least recently used one is dropped
    "version_check_seconds": 5,      # how often the registry is read again for a task
    "lakebase": {
        "instance": LAKEBASE_INSTANCE,
        "host": LAKEBASE_HOST,
        "database": LAKEBASE_DATABASE,
        "schema": SCHEMA,
        "rows_table": ROWS_SYNCED_TABLE,
        "registry_table": REGISTRY_SYNCED_TABLE,
    },
}
settings_path = "/tmp/kumo_lakebase_context_settings.json"
with open(settings_path, "w") as fh:
    json.dump(settings, fh, indent=1)

# local_dir gives plain files; the default Hugging Face cache holds symlinks, which MLflow does not accept
weights = dict(repo_id="nvidia/Kumo-Tabular", revision="v1.0.1", local_dir="/tmp/kumo_tabular_weights")
artifacts = {
    "settings": settings_path,
    "classification": hf_hub_download(filename=f"{MODEL_SIZE}/classifier.pt", **weights),
    "regression": hf_hub_download(filename=f"{MODEL_SIZE}/regressor.pt", **weights),
}

# Fixed output schema for every task; the input is free, because every task has its own columns
output_schema = Schema([
    ColSpec(DataType.string, "prediction"),
    ColSpec(DataType.string, "details"),
    ColSpec(DataType.string, "task"),
    ColSpec(DataType.long, "context_version"),
    ColSpec(DataType.boolean, "context_cached"),
    ColSpec(DataType.double, "server_ms"),
])
signature = ModelSignature(inputs=None, outputs=output_schema,
                           params=ParamSchema([ParamSpec("task", DataType.string, "")]))

# The serving container: Python and the model library are pinned, whatever this notebook runs on
conda_env = {
    "name": "kumo-tabular-lakebase-context",
    "channels": ["conda-forge"],
    "dependencies": ["python=3.12", "pip", {"pip": [
        "structured-data-models==0.2.1",
        "torch>=2.7,<2.8",
        "psycopg[binary]>=3.2",
        "databricks-sdk>=0.81.0",
        "pandas>=2.2",
        "numpy>=1.26",
        f"mlflow=={mlflow.__version__}",
    ]}],
}

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME_SERVING_ENDPOINTS)
with mlflow.start_run(run_name="kumo_tabular_lakebase_context") as run:
    model_info = mlflow.pyfunc.log_model(
        name="kumo_tabular_lakebase_context",
        python_model="/tmp/kumo_lakebase_context_model.py",
        artifacts=artifacts,
        signature=signature,
        conda_env=conda_env,
        resources=[DatabricksLakebase(database_instance_name=LAKEBASE_INSTANCE)],
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Model URI: {model_info.model_uri}")
    print(f"Registered as: {REGISTERED_MODEL_NAME} version {model_info.registered_model_version}")

### Create GPU serving endpoint

In [0]:
latest_version = str(model_info.registered_model_version)
served_entity = ServedEntityInput(
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=latest_version,
    workload_size="Small",
    workload_type=ServingModelWorkloadType.GPU_SMALL,
    scale_to_zero_enabled=True,
)

# Postgres roles before the endpoint exists, to find the role that Model Serving adds for it
roles_before = {r.name for r in w.database.list_database_instance_roles(LAKEBASE_INSTANCE)}

print(f"Deploying model version {latest_version} of '{REGISTERED_MODEL_NAME}'")
try:
    w.serving_endpoints.get(SERVING_ENDPOINT)
    w.serving_endpoints.update_config(name=SERVING_ENDPOINT, served_entities=[served_entity])
    print(f"Endpoint '{SERVING_ENDPOINT}' update triggered.")
except NotFound:
    w.serving_endpoints.create(
        name=SERVING_ENDPOINT,
        config=EndpointCoreConfigInput(name=SERVING_ENDPOINT, served_entities=[served_entity]),
    )
    print(f"Endpoint '{SERVING_ENDPOINT}' creation triggered.")

In [0]:
# Poll until the serving endpoint is ready and serves the new version
print(f"Waiting for endpoint '{SERVING_ENDPOINT}' to be ready...")
while True:
    endpoint = w.serving_endpoints.get(SERVING_ENDPOINT)
    state = endpoint.state
    served = {e.entity_version for e in (endpoint.config.served_entities if endpoint.config else [])}
    print(f"  State: ready={state.ready} | config_update={state.config_update} | versions={sorted(served)}")
    if state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
        raise RuntimeError(f"Endpoint deployment failed: {state}")
    if (state.ready == EndpointStateReady.READY
            and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING
            and latest_version in served):
        print(f"\nEndpoint '{SERVING_ENDPOINT}' is ready!")
        break
    time.sleep(120)

### Let the endpoint read the context tables

Model Serving signs the endpoint in to Lakebase with a Postgres role of its own. That role can connect, but it
cannot read the two synced tables until it is granted access. The cell finds the roles that appeared while the
endpoint was deployed and grants them read access.

Model Serving creates a new role for every model version it deploys. Run this cell again after every update of
the endpoint.

In [0]:
roles = [r.name for r in w.database.list_database_instance_roles(LAKEBASE_INSTANCE)]
endpoint_roles = [r for r in roles if r not in roles_before and r != current_user]
for role in endpoint_roles:
    lakebase_sql(f'GRANT USAGE ON SCHEMA "{SCHEMA}" TO "{role}"', fetch=False)
    lakebase_sql(
        f'GRANT SELECT ON "{SCHEMA}"."{ROWS_SYNCED_TABLE}", "{SCHEMA}"."{REGISTRY_SYNCED_TABLE}" TO "{role}"',
        fetch=False,
    )
    print(f"Granted read access to {role}")
if not endpoint_roles:
    print("No new Postgres role appeared. If a request below fails with 'permission denied', grant read access "
          "on the two synced tables to the role named in the error.")

## Inference for five tasks from one endpoint

For each task the test rows are sent twice. The first request makes the endpoint fetch the context from
Lakebase and fit it. The second request finds the fitted context on the GPU. The metrics are logged to the
shared results table, like in the other KumoTabular notebooks.

In [0]:
def predict(task, X, retries=5):
    """Send rows to the endpoint for one task. Returns the answer and the round-trip time in ms."""
    records = [{**row, "__task__": task} for row in to_records(X)]
    for attempt in range(retries):
        start = time.time()
        try:
            resp = w.serving_endpoints.query(name=SERVING_ENDPOINT, dataframe_records=records)
            return pd.DataFrame(resp.predictions), (time.time() - start) * 1000
        except Exception as e:  # an endpoint that scaled to zero needs a few minutes to wake up
            if attempt == retries - 1:
                raise
            print(f"  retry in 60s: {str(e)[:200]}")
            time.sleep(60)


def task_metrics(task, answer):
    X_train, X_test, y_train, y_test = splits[task]
    if TASKS[task]["problem_type"] == "regression":
        return regression_metrics(y_test, answer["prediction"].astype(float).values)
    labels = sorted(y_train.unique())
    y_pred = answer["prediction"].astype(y_test.dtype).values
    y_proba = np.array([[json.loads(d)[str(c)] for c in labels] for d in answer["details"]])
    return classification_metrics(y_test, y_pred, y_proba)

In [0]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

summary = []
for task, t in TASKS.items():
    X_train, X_test, y_train, y_test = splits[task]
    first, first_ms = predict(task, X_test)      # fetches the context from Lakebase and fits it
    second, second_ms = predict(task, X_test)    # the fitted context is already on the GPU
    assert second["context_cached"].all() and (first["prediction"] == second["prediction"]).all()
    metrics = task_metrics(task, second)

    with mlflow.start_run(run_name=f"{task}_kumo_serving_lakebase_context"):
        mlflow.log_params({
            "vendor": "kumo", "model_type": "KumoTabularLakebaseContext",
            "task": task, "problem_type": t["problem_type"],
            "n_features": X_train.shape[1], "train_samples": len(X_train), "test_samples": len(X_test),
            "serving_endpoint": SERVING_ENDPOINT, "context_version": int(second["context_version"][0]),
        })
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="kumo", task=task, problem_type=t["problem_type"],
                   model_name="KumoTabular (serving, context in Lakebase)", metrics=metrics,
                   n_train=len(X_train), n_test=len(X_test), n_features=X_train.shape[1])

    summary.append({
        "task": task,
        "problem_type": t["problem_type"],
        "context_rows": len(X_train),
        "test_rows": len(X_test),
        "first_request_ms": round(first_ms),
        "first_request_server_ms": float(first["server_ms"][0]),
        "next_request_ms": round(second_ms),
        "next_request_server_ms": float(second["server_ms"][0]),
        **{k: round(v, 4) for k, v in metrics.items() if v is not None},
    })
    print(summary[-1])

display(pd.DataFrame(summary))

A single row is the usual real-time case. The cell below sends one row at a time to a task whose context is
already fitted.

In [0]:
task = "supplier_delay_risk"
X_test = splits[task][1]
single = [predict(task, X_test.iloc[[i]]) for i in range(20)]
round_trip = np.array([ms for _, ms in single])
server = np.array([float(answer["server_ms"][0]) for answer, _ in single])
print(f"One row, {task}: round trip p50 {np.percentile(round_trip, 50):.0f} ms, p95 {np.percentile(round_trip, 95):.0f} ms; "
      f"inside the model p50 {np.percentile(server, 50):.0f} ms")
single[0][0]

## Refresh a context without a new deployment

New labelled rows arrive for the supplier delay task. Here they come from the `supplier_delay_risk_score`
table. The steps are:

1. Write the rows of version 2 to the Delta table and sync it.
2. Point the registry at version 2 and sync it.

The endpoint notices the new version on its next registry read and fits it on the next request. The model
version and the endpoint stay as they are. The rows of version 1 stay in the table, so setting the registry
back to version 1 rolls the change back.

In [0]:
task = "supplier_delay_risk"
target = TASKS[task]["target"]
X_train, X_test, y_train, y_test = splits[task]
new_rows = spark.table("supplier_delay_risk_score").toPandas()

rows_v2, registry_v2 = context_frames(
    task, 2,
    pd.concat([X_train, new_rows.drop(columns=[target])], ignore_index=True),
    pd.concat([y_train, new_rows[target]], ignore_index=True),
    TASKS[task]["problem_type"],
)

start = time.time()
spark.sql(f"DELETE FROM {CONTEXT_ROWS_TABLE} WHERE task = '{task}' AND version = 2")   # safe to run again
spark.createDataFrame(rows_v2).write.mode("append").saveAsTable(CONTEXT_ROWS_TABLE)
sync_now(ROWS_SYNCED)

spark.createDataFrame(registry_v2).createOrReplaceTempView("registry_update")
spark.sql(f"""
    MERGE INTO {CONTEXT_REGISTRY_TABLE} t USING registry_update s ON t.task = s.task
    WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
""")
sync_now(REGISTRY_SYNCED)
display(wait_for_rows())
print(f"Version 2 of '{task}' reached Lakebase {time.time() - start:.0f}s after the write started")

In [0]:
time.sleep(settings["version_check_seconds"] + 1)   # let the endpoint read the registry again

first, first_ms = predict(task, X_test)
second, second_ms = predict(task, X_test)
assert (first["context_version"] == 2).all() and not first["context_cached"].any() and second["context_cached"].all()

print(f"context version {int(first['context_version'][0])}, {len(rows_v2)} context rows")
print(f"first request after the switch: {first_ms:.0f} ms (fetch and fit); next request: {second_ms:.0f} ms")
print("metrics with version 2:", {k: round(v, 4) for k, v in task_metrics(task, second).items() if v is not None})

## Inference example with ai_query

[ai_query](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) cannot pass `params`, so
the task goes in the `__task__` column. The rows carry only the features of the transactions to score. The
context stays in Lakebase.

In [0]:
%sql
SELECT
  t.*,
  result.prediction,
  result.details,
  result.context_version
FROM (
  SELECT *,
    ai_query(
      'kumo_tabular_lakebase_context',
      struct(*, 'supplier_delay_risk' AS __task__),
      returnType => 'STRUCT<prediction:STRING, details:STRING, task:STRING, context_version:BIGINT, context_cached:BOOLEAN, server_ms:DOUBLE>'
    ) AS result
  FROM (
    SELECT * EXCEPT (is_delayed) FROM supplier_delay_risk_score LIMIT 10
  )
) t

## Results

All rows for this run are in the shared results table. The comparison notebook
aggregates across vendors.

In [0]:
display(
    spark.table(RESULTS_TABLE)
         .where("vendor = 'kumo'")
         .orderBy("task", "model_name", "run_ts")
)

## What to expect, and the limits

**Measured with this notebook** on a `GPU_SMALL` endpoint (one T4), with contexts of 840 to 1,920 rows. The
times are spent inside the model. A round trip from a notebook in the same region added 50 to 150 ms.

| Setting | First request for a task (fetch and fit) | Later request with 320 to 480 rows | Later request with one row |
|---|---|---|---|
| `MODEL_SIZE = "large"`, `NUM_ESTIMATORS = 8`, the setting of this notebook and of the other KumoTabular notebooks | 3.4 to 6.5 s | 0.8 to 1.5 s | 270 ms |
| `MODEL_SIZE = "small"`, `NUM_ESTIMATORS = 1` | 0.2 to 0.4 s | 55 to 95 ms | 40 ms |

- The very first request to a new endpoint also opens the connection to Lakebase. It took 7.0 s with the first
  setting and 2.1 s with the second.
- On these five tasks the two settings gave the same accuracy, ROC-AUC, F1 and R2 to within 0.02. Check this
  on your own data before you choose the smaller setting.
- A switch to a new context version took about two minutes, for the two triggered syncs. The endpoint was not
  updated.

**Larger contexts.** The fetch from Lakebase stays the small part of the first request. This table is from a
separate test on the same kind of endpoint, with the small model, one estimator and a card-fraud table with 24
columns. "Baked" means that the context is stored in the model artifact and fitted when the endpoint starts.

| Context rows | First request: fetch | First request: encode and fit | Later requests, context in Lakebase | Later requests, context baked |
|---|---|---|---|---|
| 1,024 | 11 ms | 0.2 s | 58 ms | 56 ms |
| 8,192 | 55 ms | 1.3 s | 80 ms | 82 ms |
| 32,768 | 168 ms | 11.0 s | 120 ms | 117 ms |

- Once a context is fitted, a request costs the same as with a baked context. The registry read adds about
  3 ms, and it happens at most once every `version_check_seconds`.
- Every replica fits a context on its own first request, and again after the endpoint has scaled to zero.
  Send one request per task after a version switch if the first caller should not wait.

**Limits**

- Fitted contexts use GPU memory. The model keeps `max_cached_tasks` of them and drops the one that was used
  longest ago. Choose the number for your GPU and your context sizes.
- A request still has to stay under the 16 MB payload limit, but it now carries only the rows to predict.
- The endpoint needs a Lakebase database instance that the MLflow resource `DatabricksLakebase` accepts, and
  the user who creates the endpoint needs the `databricks_superuser` role on it.
- Every endpoint update brings a new Postgres role, which needs read access to the two synced tables.
- The context rows are fitted as they are. Any sampling, e.g., more rows of a rare class, has to be done when
  the context version is written.
- The numbers above are from one GPU, one replica and one request at a time.

## Clean up

Set `CLEAN_UP = True` to delete the endpoint, the two synced tables with their Postgres copies, the two Delta
tables and the registered model.

In [0]:
CLEAN_UP = False

if CLEAN_UP:
    try:
        w.serving_endpoints.delete(SERVING_ENDPOINT)
        print(f"Deleted endpoint {SERVING_ENDPOINT}")
    except NotFound:
        pass
    for name in (ROWS_SYNCED, REGISTRY_SYNCED):
        try:
            w.database.delete_synced_database_table(name)
            print(f"Deleted synced table {name}")
        except NotFound:
            pass
    for table in (ROWS_SYNCED_TABLE, REGISTRY_SYNCED_TABLE):   # Lakebase drops synced tables one at a time
        lakebase_sql(f'DROP TABLE IF EXISTS "{SCHEMA}"."{table}"', fetch=False)
    for table in (CONTEXT_ROWS_TABLE, CONTEXT_REGISTRY_TABLE):
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    client = mlflow.MlflowClient()
    for version in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'"):
        client.delete_model_version(REGISTERED_MODEL_NAME, version.version)
    client.delete_registered_model(REGISTERED_MODEL_NAME)
    print("Dropped the context tables and the registered model")